Vamos a realizar una copia de seguridad en mi servidor personal de PostgreSQL y actualizar el servidor db de SQLITE del proyecto, cuando confirmemos que los archivos han sido guardados satisfactoriamente y con los formatos deseados.
Borraremos todos los archivos innecesarios.
Para ello creamos un .venv con las librerias indispensables.
'pip install psycopg pandas sqlalchemy numpy'
Para ello haremos una funcion que itere sobre las carpetas donde se alojan los .csv.

In [1]:
#Creamos un script para transferir todos los csv del proyecto a la base de datos local del repositorio sqlite3. Cerciorandonos de su funcionamiento.
import pandas as pd
import sqlite3
import os


carpetas_csv = [
    "C:/Users/Josue/4GA.DataScience/data/processed/",
    "C:/Users/Josue/4GA.DataScience/data/raw/",
    'C:/Users/Josue/4GA.DataScience/src/PASO5'
]
ruta_db_sqlite = "C:/Users/Josue/4GA.DataScience/src/EcoUE.db"


conexion = sqlite3.connect(ruta_db_sqlite)
cursor = conexion.cursor()


for carpeta in carpetas_csv:
    
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta_archivo = os.path.join(carpeta, archivo)
            nombre_tabla = os.path.splitext(archivo)[0]

            try:
                
                df = pd.read_csv(ruta_archivo)

                
                df.to_sql(nombre_tabla, conexion, if_exists="replace", index=False)

                print(f"Archivo {archivo} transferido a la tabla {nombre_tabla}")

            except Exception as e:
                print(f"Error al procesar el archivo {archivo}: {e}")


conexion.close()

Archivo dfeda.csv transferido a la tabla dfeda
Archivo df_cntrytype.csv transferido a la tabla df_cntrytype
Archivo UE128k.csv transferido a la tabla UE128k
Archivo Grupo0.csv transferido a la tabla Grupo0
Archivo Grupo1.csv transferido a la tabla Grupo1
Archivo Grupo2.csv transferido a la tabla Grupo2
Archivo UEmedG1.csv transferido a la tabla UEmedG1
Archivo UEporG2.csv transferido a la tabla UEporG2
Archivo UEricG0.csv transferido a la tabla UEricG0


Ahora lo mismo para almacenar los csv como tablas en nuestro servidor propio, tendremos que tener cuidado con el archivo configsql.cfg deberemos tenerlo apuntado en el gitignore para no revelar el archivo en el repositorio
en remoto de github.

In [2]:
import pandas as pd
import psycopg
import os
import configparser

config = configparser.ConfigParser()
config.read('C:/Users/Josue/4GA.DataScience/configsql.cfg')
carpetas_csv = [
    "C:/Users/Josue/4GA.DataScience/data/processed/",
    "C:/Users/Josue/4GA.DataScience/data/raw/",
    'C:/Users/Josue/4GA.DataScience/src/PASO5'
]

db_config = dict(config['postgresql'])
dataframes = {}

# Fase 1: Cargar todos los CSV en DataFrames
for carpeta in carpetas_csv:
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta_archivo = os.path.join(carpeta, archivo)
            nombre_tabla = os.path.splitext(archivo)[0]
            try:
                df = pd.read_csv(ruta_archivo)
                dataframes[nombre_tabla] = df
                print(f"Archivo {archivo} cargado en DataFrame '{nombre_tabla}'")
            except pd.errors.EmptyDataError:
                print(f"Advertencia: El archivo CSV {archivo} está vacío y no se cargó.")
            except FileNotFoundError:
                print(f"Error: No se encontró el archivo {ruta_archivo}")
            except Exception as e_cargar:
                print(f"Error al cargar el archivo {archivo} en DataFrame: {e_cargar}")

# Fase 2: Conectar a PostgreSQL y transferir los DataFrames
try:
    conn = psycopg.connect(
        host=db_config['host'],
        dbname=db_config['dbname'],
        user=db_config['user'],
        password=db_config['password'],
        port=db_config['port'],
        # sslmode='require',
        connect_timeout=10
    )
    cur = conn.cursor()

    for nombre_tabla, df in dataframes.items():
        try:
            columnas_csv = df.columns.tolist()
            columnas_str = ", ".join([f'"{col}" TEXT' for col in columnas_csv])

            # Crear la tabla SI NO EXISTE con las columnas del DataFrame
            cur.execute(f'CREATE TABLE IF NOT EXISTS "{nombre_tabla}" ({columnas_str});')
            conn.commit()

            # Insertar los datos en la tabla
            for index, row in df.iterrows():
                placeholders = ", ".join(['%s'] * len(columnas_csv))
                insert_query = f'INSERT INTO "{nombre_tabla}" VALUES ({placeholders});'
                try:
                    cur.execute(insert_query, row.values.tolist())
                except psycopg.Error as e_insert:
                    conn.rollback()
                    print(f"Error al insertar fila en {nombre_tabla}: {e_insert}")
                    print(f"Fila problemática (índice {index}): {row.values}")
                    break # Detener el procesamiento de este DataFrame si hay errores de inserción

            conn.commit()
            print(f"DataFrame '{nombre_tabla}' transferido a la tabla '{nombre_tabla}' en PostgreSQL")

        except Exception as e_transferir:
            conn.rollback()
            print(f"Error al transferir el DataFrame '{nombre_tabla}' a PostgreSQL: {e_transferir}")

    cur.close()
    conn.close()

except psycopg.Error as e_conexion:
    print(f"Error al conectar a PostgreSQL: {e_conexion}")
finally:
    if 'conn' in locals() and conn:
        conn.close()
        print("Conexión a PostgreSQL cerrada.")

Archivo dfeda.csv cargado en DataFrame 'dfeda'
Archivo df_cntrytype.csv cargado en DataFrame 'df_cntrytype'
Archivo UE128k.csv cargado en DataFrame 'UE128k'
Archivo Grupo0.csv cargado en DataFrame 'Grupo0'
Archivo Grupo1.csv cargado en DataFrame 'Grupo1'
Archivo Grupo2.csv cargado en DataFrame 'Grupo2'
Archivo UEmedG1.csv cargado en DataFrame 'UEmedG1'
Archivo UEporG2.csv cargado en DataFrame 'UEporG2'
Archivo UEricG0.csv cargado en DataFrame 'UEricG0'
Error al insertar fila en dfeda: INSERT tiene más expresiones que columnas de destino
LINE 1: ...$14, $15, $16, $17, $18, $19, $20, $21, $22, $23, $24, $25, ...
                                                             ^
Fila problemática (índice 0): ['Bélgica' 7.0 7.0 7.0 4.0 5.0 3.0 5.0 4.0 4.0 7.0 4.0 7.0 5.0 4.0 5.0 7.0
 4.0 5.0 8.0 2.0 0.0 0 7.75 0.0 0]
DataFrame 'dfeda' transferido a la tabla 'dfeda' en PostgreSQL
DataFrame 'df_cntrytype' transferido a la tabla 'df_cntrytype' en PostgreSQL
Error al insertar fila en UE128k: INSER

HEMOS DESACTIVADO LA OPCION DE CONECTARNOS MEDIANTE SSL Y SSH, PUESTO QUE MI SERVIDOR SE ALOJA EN ESTE MISMO 
ORDENADOR Y NO ES NECESARIO. HAREMOS UNA PRUEBA DE CARGA DE TABLAS A DATAFRAMES PAR CONFIRMAR LA CONEXION Y EL BUEN ALMACENAMIENTO DE LOS DATOS.

Siguiente documento del proyecto en PASO4.